# Oceanographic forcing for the Cox's Bazar–Teknaf coast (1993–2023)

1. Edit `GEE_PROJECT` in the CONFIG cell (the one that starts with `GEE_PROJECT = ...`) — nothing else needs editing.
2. Run everything: **Runtime > Run all**.
3. Approve the Earth Engine sign-in when the pop-up appears.
4. Outputs go to `/content/ocean_outputs` and are downloaded as `ocean_outputs.zip` at the end — send the zip and the printed SUMMARY to Claude.

In [ ]:
!pip -q install cartopy

In [ ]:
# CONFIG -- the only cell you need to edit
import os

GEE_PROJECT = "your-gee-project-id"   # <-- EDIT ME
PERIOD_START, PERIOD_END = "1993-01-01", "2023-12-31"
YEAR_START, YEAR_END = 1993, 2023
OUT_DIR = "/content/ocean_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DOMAIN = dict(lon_min=91.3, lon_max=92.5, lat_min=20.4, lat_max=21.8)   # HYCOM map/sampling box
WAVE_LATS = [20.0, 20.5, 21.0, 21.5]; WAVE_LONS = [90.5, 91.0, 91.5, 92.0]   # ERA5-Ocean 0.5 deg points

COAST_AZ = 335.0         # alongshore direction (Teknaf -> Cox's Bazar), degrees clockwise from north
SEAWARD_NORMAL = 245.0   # direction pointing offshore (WSW)
OFFSHORE_KM = 10.0       # forcing is sampled this far offshore of each zone

# Hotspot zones from the two-epoch shoreline analysis (EPR 1992-2023, m/yr).
# Coordinates are approximate shoreline locations (accuracy ~1-3 km), adequate for 8 km HYCOM and 50 km ERA5 grids.
ZONES = {
    "A1 Nazirartek":                    dict(lat=21.468, lon=91.958, epr=+13.74, kind="Accretion"),
    "E3 Flight Deck BAF / Somiti Para": dict(lat=21.445, lon=91.962, epr=-4.80,  kind="Erosion"),
    "A3 Sonar Para / BORI Beach":       dict(lat=21.330, lon=92.020, epr=+5.59,  kind="Accretion"),
    "A2 Reju Khal":                     dict(lat=21.262, lon=92.045, epr=+7.88,  kind="Accretion"),
    "E2 Sabrang":                       dict(lat=20.805, lon=92.285, epr=-4.85,  kind="Erosion"),
    "E1 Dakhinpara":                    dict(lat=20.762, lon=92.318, epr=-24.46, kind="Erosion"),
}

# (start_month, end_month); Dec-Feb wraps the year
SEASONS = {"SW monsoon (Jun-Sep)": (6, 9), "NE monsoon (Dec-Feb)": (12, 2), "Annual": (1, 12)}
HYCOM_THIN_DAYS = [1, 6, 11, 16, 21, 26]   # HYCOM means use these days of each month (fast, statistically ample); None = all days
RUN_CYCLONES = True

In [ ]:
# HELPERS: imports, geometry, trend statistics and map utilities
import io, json, time, shutil
import numpy as np
import pandas as pd
import requests
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from IPython.display import display

# cartopy is optional: if anything fails we draw on plain matplotlib axes (no coastline)
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.io import shapereader
    shapereader.natural_earth(resolution="10m", category="physical", name="land")        # download test
    shapereader.natural_earth(resolution="10m", category="physical", name="coastline")
    HAS_CARTOPY = True
except Exception as err:
    HAS_CARTOPY = False
    print("cartopy not usable -> plain axes without coastline:", err)
TR = dict(transform=ccrs.PlateCarree()) if HAS_CARTOPY else {}   # pass to every plotting call

SHORT = dict(zip(SEASONS, ["sw", "ne", "ann"]))   # short season names used in file names / column suffixes
R_EARTH = 6371.0                                  # km


def months_in(season):
    """List of month numbers of a season (handles Dec-Feb wrap-around)."""
    start, end = SEASONS[season]
    return list(range(start, end + 1)) if start <= end else list(range(start, 13)) + list(range(1, end + 1))


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (works with numpy arrays)."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = p2 - p1, np.radians(lon2) - np.radians(lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))


def destination(lat, lon, azimuth_deg, dist_km):
    """Point reached from (lat, lon) after dist_km along azimuth_deg (spherical great-circle formula)."""
    phi1, lam1 = np.radians(lat), np.radians(lon)
    az, d = np.radians(azimuth_deg), dist_km / R_EARTH
    phi2 = np.arcsin(np.sin(phi1) * np.cos(d) + np.cos(phi1) * np.sin(d) * np.cos(az))
    lam2 = lam1 + np.arctan2(np.sin(az) * np.sin(d) * np.cos(phi1), np.cos(d) - np.sin(phi1) * np.sin(phi2))
    return float(np.degrees(phi2)), float((np.degrees(lam2) + 540) % 360 - 180)


def unit(az):
    """(east, north) components of a unit vector pointing along azimuth az (degrees from north)."""
    return np.sin(np.radians(az)), np.cos(np.radians(az))


T_E, T_N = unit(COAST_AZ)                        # alongshore unit vector (towards NNW)
S_E, S_N = unit((SEAWARD_NORMAL + 180) % 360)    # shoreward unit vector (pointing onshore)

# offshore sampling point and short label (E1, A1, ...) for every zone
for name, z in ZONES.items():
    z["off_lat"], z["off_lon"] = destination(z["lat"], z["lon"], SEAWARD_NORMAL, OFFSHORE_KM)
    z["label"] = name.split()[0]


def find_col(columns, prefix):
    """Column whose name starts with prefix (Open-Meteo may append a model suffix or unit)."""
    for c in columns:
        if str(c).startswith(prefix):
            return c
    raise KeyError(f"no column starting with {prefix}")


def to_wave_df(raw):
    """Tidy an Open-Meteo hourly table into columns time, hs, dir, tm (NaN rows dropped)."""
    out = pd.DataFrame({
        "time": pd.to_datetime(raw[find_col(raw.columns, "time")]),
        "hs": pd.to_numeric(raw[find_col(raw.columns, "wave_height")], errors="coerce"),
        "dir": pd.to_numeric(raw[find_col(raw.columns, "wave_direction")], errors="coerce"),   # direction waves come FROM
        "tm": pd.to_numeric(raw[find_col(raw.columns, "wave_period")], errors="coerce"),
    })
    return out.dropna().reset_index(drop=True)


def trend_stats(years, values):
    """Mann-Kendall test (Kendall tau vs time) and Sen's slope; slopes are returned per decade."""
    years, values = np.asarray(years, float), np.asarray(values, float)
    tau, p = stats.kendalltau(years, values)
    slope, intercept, lo, hi = stats.theilslopes(values, years, 0.95)
    return dict(slope_per_decade=float(slope * 10), ci95_per_decade=[float(lo * 10), float(hi * 10)],
                tau=float(tau), p=float(p), mean=float(values.mean()), n_years=int(len(years)),
                intercept=float(intercept))


def map_axes(n_panels, extent, figsize):
    """Row of map panels. extent = (lon_min, lon_max, lat_min, lat_max). Falls back to plain axes."""
    kw = dict(subplot_kw=dict(projection=ccrs.PlateCarree())) if HAS_CARTOPY else {}
    fig, axes = plt.subplots(1, n_panels, figsize=figsize, **kw)
    axes = np.atleast_1d(axes)
    lon0, lon1, lat0, lat1 = extent
    for ax in axes:
        if HAS_CARTOPY:
            ax.set_extent(extent, crs=ccrs.PlateCarree())
            ax.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m", facecolor="#f2e8d0",
                                                        edgecolor="none"), zorder=0)
            ax.add_feature(cfeature.NaturalEarthFeature("physical", "coastline", "10m", facecolor="none",
                                                        edgecolor="black"), linewidth=0.6, zorder=3)
            gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.6, linestyle="--")
            gl.top_labels = False      # labels on left / bottom only
            gl.right_labels = False
        else:
            ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
            ax.set_aspect(1 / np.cos(np.radians((lat0 + lat1) / 2)))
            ax.grid(True, lw=0.3, ls="--", color="gray")
            ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
    return fig, axes


def add_zones(ax, legend=False):
    """Hotspot zones: red = erosion, green = accretion, with short bold labels."""
    for z in ZONES.values():
        colour = "red" if z["kind"] == "Erosion" else "limegreen"
        ax.scatter(z["lon"], z["lat"], s=45, c=colour, edgecolors="black", zorder=5, **TR)
        dy = -0.02 if z["kind"] == "Erosion" else 0.02      # keeps close zones (A1/E3) apart
        ax.text(z["lon"] + 0.03, z["lat"] + dy, z["label"], fontsize=8, fontweight="bold",
                va="center", zorder=6, **TR)
    if legend:
        ax.scatter([], [], s=45, c="red", edgecolors="black", label="Erosion zone")
        ax.scatter([], [], s=45, c="limegreen", edgecolors="black", label="Accretion zone")
        ax.legend(loc="lower left", fontsize=7, framealpha=0.9)


def save_fig(fig, name):
    """Save a figure at 300 dpi into OUT_DIR, then show it."""
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300, bbox_inches="tight")
    plt.show()
    print("saved", name)

In [ ]:
# EARTH ENGINE initialisation
import ee

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()          # opens the sign-in pop-up; approve it
    ee.Initialize(project=GEE_PROJECT)
print("Earth Engine OK")

In [ ]:
# HYCOM seasonal climatology: surface currents + sea surface temperature
VEL = ee.ImageCollection("HYCOM/sea_water_velocity")   # velocity_u_0 / velocity_v_0, stored integers x 0.001 = m/s
TMP = ee.ImageCollection("HYCOM/sea_temp_salinity")    # water_temp_0, value x 0.001 + 20 = degC
VEL_PROJ = VEL.first().select("velocity_u_0").projection()   # native 0.08 deg grid
REGION = ee.Geometry.Rectangle([DOMAIN["lon_min"], DOMAIN["lat_min"], DOMAIN["lon_max"], DOMAIN["lat_max"]])


def hycom_filter(col, d0, d1, start_m, end_m):
    """Dates + calendar-month filter (start_m > end_m wraps the year) + optional day-of-month thinning."""
    col = col.filterDate(d0, d1).filter(ee.Filter.calendarRange(start_m, end_m, "month"))
    if HYCOM_THIN_DAYS:
        col = col.filter(ee.Filter.Or(*[ee.Filter.calendarRange(d, d, "day_of_month") for d in HYCOM_THIN_DAYS]))
    return col


def fc_to_df(fc, names):
    """Earth Engine FeatureCollection (sample result) -> DataFrame with lon, lat and the named properties."""
    rows = []
    for f in fc["features"]:
        lon, lat = f["geometry"]["coordinates"]
        row = {"lon": round(lon, 4), "lat": round(lat, 4)}
        for n in names:
            row[n] = f["properties"].get(n)
        rows.append(row)
    return pd.DataFrame(rows, columns=["lon", "lat"] + list(names))


def sample_season(season, d0, d1):
    """Mean u, v, sst of one season over [d0, d1], sampled on the HYCOM grid."""
    start_m, end_m = SEASONS[season]
    # scaling is applied AFTER the mean
    vel = (hycom_filter(VEL, d0, d1, start_m, end_m).select(["velocity_u_0", "velocity_v_0"])
           .mean().multiply(0.001).rename(["u", "v"]))
    sst = (hycom_filter(TMP, d0, d1, start_m, end_m).select("water_temp_0")
           .mean().multiply(0.001).add(20).rename("sst"))
    v_df = fc_to_df(vel.sample(region=REGION, projection=VEL_PROJ, geometries=True).getInfo(), ["u", "v"])
    t_df = fc_to_df(sst.sample(region=REGION, projection=VEL_PROJ, geometries=True).getInfo(), ["sst"])
    return v_df.merge(t_df, on=["lon", "lat"], how="outer")


HYCOM = {}   # season -> DataFrame (lon, lat, u, v, sst, speed, along)
for season in SEASONS:
    try:
        df = sample_season(season, PERIOD_START, PERIOD_END)
    except ee.EEException as err:
        if not any(w in str(err).lower() for w in ("timed out", "memory", "too many")):
            raise
        # fallback: seasonal mean year by year, then average the yearly samples
        print(f"{season}: Earth Engine timed out -> using the year-by-year fallback (slower)")
        yearly = []
        for y in range(YEAR_START, YEAR_END + 1):
            try:
                yearly.append(sample_season(season, f"{y}-01-01", f"{y}-12-31"))
            except Exception as e2:
                print("  skipped year", y, "-", str(e2)[:80])
        df = pd.concat(yearly).groupby(["lon", "lat"], as_index=False).mean()
    df = df.dropna(subset=["u", "v", "sst"]).reset_index(drop=True)
    df["speed"] = np.hypot(df["u"], df["v"])
    df["along"] = df["u"] * T_E + df["v"] * T_N      # >0 flow toward NNW (Cox's Bazar), <0 toward SSE (Teknaf)
    HYCOM[season] = df
    df.to_csv(os.path.join(OUT_DIR, f"hycom_{SHORT[season]}.csv"), index=False)
    print(f"{season}: {len(df)} cells | mean speed {df.speed.mean():.3f} m/s | max speed {df.speed.max():.3f} m/s "
          f"| mean SST {df.sst.mean():.2f} degC")

In [ ]:
# HYCOM maps: currents (a) and SST (b), one panel per season
EXTENT = (DOMAIN["lon_min"], DOMAIN["lon_max"], DOMAIN["lat_min"], DOMAIN["lat_max"])


def hycom_map(var, cmap, vmin, vmax, cbar_label, title, fname, arrows):
    fig, axes = map_axes(len(SEASONS), EXTENT, (15, 6.5))
    for i, (ax, season) in enumerate(zip(axes, SEASONS)):
        df = HYCOM[season]
        grid = df.pivot_table(index="lat", columns="lon", values=var)
        mesh = ax.pcolormesh(grid.columns.values, grid.index.values, grid.values, cmap=cmap,
                             vmin=vmin, vmax=vmax, shading="nearest", **TR)
        if arrows:   # white arrows = direction and relative strength of the current
            ax.quiver(df["lon"], df["lat"], df["u"], df["v"], color="white", scale=4, scale_units="width",
                      width=0.003, zorder=4, **TR)
        add_zones(ax, legend=(i == 0))
        ax.set_title(season, fontsize=11)
    fig.colorbar(ScalarMappable(norm=Normalize(vmin, vmax), cmap=cmap), ax=list(axes),
                 orientation="horizontal", fraction=0.05, pad=0.10, label=cbar_label)
    fig.suptitle(title, fontsize=13)
    save_fig(fig, fname)


# (a) surface current speed; one shared colour scale from 0 to the 98th percentile of all seasons
all_speed = np.concatenate([HYCOM[s]["speed"].values for s in SEASONS])
hycom_map("speed", "viridis", 0, np.percentile(all_speed, 98), "Surface current speed (m s⁻¹)",
          "HYCOM mean surface currents, 1993–2023", "Fig_currents_seasonal.png", arrows=True)

# (b) sea surface temperature; shared scale from the 2nd to the 98th percentile
all_sst = np.concatenate([HYCOM[s]["sst"].values for s in SEASONS])
hycom_map("sst", "RdYlBu_r", np.percentile(all_sst, 2), np.percentile(all_sst, 98), "Sea surface temperature (°C)",
          "HYCOM mean sea surface temperature, 1993–2023", "Fig_SST_seasonal.png", arrows=False)

In [ ]:
# SST trend: annual mean over the nearshore box (Mann-Kendall + Sen's slope)
BOX = ee.Geometry.Rectangle([91.8, 20.6, 92.4, 21.6])


def annual_sst(y):
    """Feature with the raw (unscaled) annual mean SST of year y over BOX; -99999 if no data."""
    y = ee.Number(y)
    col = hycom_filter(TMP, ee.Date.fromYMD(y, 1, 1), ee.Date.fromYMD(y.add(1), 1, 1), 1, 12)
    res = col.select("water_temp_0").mean().reduceRegion(ee.Reducer.mean(), BOX, 8905.6)
    return ee.Feature(None, {"year": y, "raw": ee.Number(res.get("water_temp_0", -99999))})


try:
    feats = ee.FeatureCollection(ee.List.sequence(YEAR_START, YEAR_END).map(annual_sst)).getInfo()["features"]
except ee.EEException:
    print("server-side loop failed -> requesting one year at a time")
    feats = [annual_sst(y).getInfo() for y in range(YEAR_START, YEAR_END + 1)]

sst_df = pd.DataFrame([f["properties"] for f in feats])
sst_df = sst_df[sst_df["raw"] != -99999].copy()                   # drop years without data
sst_df["sst"] = sst_df["raw"] * 0.001 + 20
sst_df = sst_df[["year", "sst"]].astype({"year": int}).reset_index(drop=True)
sst_df.to_csv(os.path.join(OUT_DIR, "sst_annual.csv"), index=False)

SST_TREND = trend_stats(sst_df["year"], sst_df["sst"])
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in SST_TREND.items() if k != "ci95_per_decade"},
      "CI95/decade:", [round(c, 3) for c in SST_TREND["ci95_per_decade"]])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sst_df["year"], sst_df["sst"], "o-", color="tab:red", label="Annual mean SST")
ax.plot(sst_df["year"], SST_TREND["intercept"] + SST_TREND["slope_per_decade"] / 10 * sst_df["year"], "k--",
        label=f"Sen slope {SST_TREND['slope_per_decade']:+.2f} °C/decade (p = {SST_TREND['p']:.3f})")
ax.set_xlabel("Year"); ax.set_ylabel("Sea surface temperature (°C)")
ax.set_title("HYCOM annual mean SST, nearshore box 91.8–92.4°E, 20.6–21.6°N")
ax.grid(alpha=0.3); ax.legend()
save_fig(fig, "Fig_SST_trend.png")

In [ ]:
# ERA5 waves via the Open-Meteo Marine API (free, no key, no account)
URL = "https://marine-api.open-meteo.com/v1/marine"
WAVE_DIR = os.path.join(OUT_DIR, "era5_waves")
os.makedirs(WAVE_DIR, exist_ok=True)
CHUNKS = [("1993-01-01", "2002-12-31"), ("2003-01-01", "2012-12-31"), ("2013-01-01", "2023-12-31")]


def get_json(params):
    """GET with retries: HTTP 429 -> wait 70 s x attempt, other errors -> wait 20 s."""
    for attempt in range(1, 7):
        try:
            r = requests.get(URL, params=params, timeout=180)
            if r.status_code == 429:
                print(f"  rate limited (429), waiting {70 * attempt} s")
                time.sleep(70 * attempt)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as err:
            print(f"  request error ({str(err)[:80]}), waiting 20 s")
            time.sleep(20)
    raise RuntimeError("Open-Meteo failed repeatedly — use Cell 9b fallback")


def download_point(lat, lon):
    """All chunks for one point -> DataFrame (time, hs, dir, tm, grid_lat, grid_lon)."""
    parts, grid = [], (np.nan, np.nan)
    for start, end in CHUNKS:
        js = get_json(dict(latitude=lat, longitude=lon, hourly="wave_height,wave_direction,wave_period",
                           models="era5_ocean", start_date=start, end_date=end,
                           cell_selection="sea", timezone="GMT"))
        grid = (js["latitude"], js["longitude"])               # grid-cell centre actually used
        raw = pd.DataFrame(js["hourly"])
        parts.append(to_wave_df(raw))
        time.sleep(8)                                          # free-tier fair use
    df = pd.concat(parts, ignore_index=True)
    df["grid_lat"], df["grid_lon"] = grid
    return df


_tmp = {}   # filled first; wave_raw is only created when everything succeeded (Cell 9b relies on this)
for lat in WAVE_LATS:
    for lon in WAVE_LONS:
        path = os.path.join(WAVE_DIR, f"pt_{lat}_{lon}.csv")
        if os.path.exists(path):                               # cached: no re-download
            df = pd.read_csv(path, parse_dates=["time"])
        else:
            print(f"downloading {lat}N {lon}E ...")
            df = download_point(lat, lon)
            df.to_csv(path, index=False)
        if df.empty:
            print(f"{lat}N {lon}E: no data returned, skipped")
            continue
        cell = (round(float(df["grid_lat"].iloc[0]), 3), round(float(df["grid_lon"].iloc[0]), 3))
        if cell in _tmp:                                       # land points snap to the nearest sea cell
            print(f"{lat}N {lon}E -> duplicate of grid cell {cell}, skipped")
            continue
        _tmp[cell] = df[["time", "hs", "dir", "tm"]].dropna().reset_index(drop=True)

wave_raw = _tmp   # dict {(grid_lat, grid_lon): DataFrame(time, hs, dir, tm)}
print(f"{len(wave_raw)} distinct ERA5 cells:", sorted(wave_raw))

In [ ]:
# CELL 9b: FALLBACK -- only used if Cell 9 fails
LOAD_UPLOADS = False

if "wave_raw" in globals() and len(wave_raw) > 0 and not LOAD_UPLOADS:
    print("Cell 9 succeeded — fallback not needed")
else:
    UPLOAD_DIR = os.path.join(OUT_DIR, "era5_waves", "upload")
    if not LOAD_UPLOADS:
        os.makedirs(UPLOAD_DIR, exist_ok=True)
        print("Open each URL in a browser (it downloads a CSV), upload the CSVs to")
        print(f"  {UPLOAD_DIR}")
        print("then set LOAD_UPLOADS = True and re-run this cell.\n")
        for lat in WAVE_LATS:
            for lon in WAVE_LONS:
                print(f"{URL}?latitude={lat}&longitude={lon}&hourly=wave_height,wave_direction,wave_period"
                      f"&models=era5_ocean&start_date={PERIOD_START}&end_date={PERIOD_END}"
                      f"&cell_selection=sea&timezone=GMT&format=csv")
    else:
        _tmp = {}
        for fname in sorted(os.listdir(UPLOAD_DIR)):
            if not fname.lower().endswith(".csv"):
                continue
            with open(os.path.join(UPLOAD_DIR, fname), encoding="utf-8") as fh:
                lines = fh.read().splitlines()
            # layout: metadata header, values line (latitude, longitude, ...), blank line, table starting with "time"
            meta = pd.read_csv(io.StringIO("\n".join(lines[:2])))
            start = next(i for i, ln in enumerate(lines) if ln.startswith("time"))
            df = to_wave_df(pd.read_csv(io.StringIO("\n".join(lines[start:]))))
            cell = (round(float(meta["latitude"].iloc[0]), 3), round(float(meta["longitude"].iloc[0]), 3))
            if cell in _tmp:
                print(fname, "-> duplicate grid cell", cell, "skipped")
                continue
            _tmp[cell] = df
        wave_raw = _tmp
        print(f"{len(wave_raw)} distinct ERA5 cells loaded:", sorted(wave_raw))

In [ ]:
# Wave statistics per ERA5 cell and season, plus pooled climatology and trend
def wave_stats(df):
    """Statistics for one cell/season. dir = direction waves come FROM (degrees)."""
    hs, tm, d = df["hs"].values, df["tm"].values, np.radians(df["dir"].values)
    w = hs ** 2 * tm                                              # energy-like weight
    dir_from = np.degrees(np.arctan2((w * np.sin(d)).sum(), (w * np.cos(d)).sum())) % 360
    # simple longshore wave-energy-flux index (proxy of the CERC relation, ~ H^2 T sin2a)
    prop = np.radians((df["dir"].values + 180) % 360)             # direction the waves travel TO
    pe, pn = np.sin(prop), np.cos(prop)
    along = pe * T_E + pn * T_N                                   # component along the coast
    shoreward = pe * S_E + pn * S_N                               # component towards the shore
    p_l = np.where(shoreward > 0, w * along * shoreward, 0.0)
    return dict(hs_mean=hs.mean(), hs_p90=np.percentile(hs, 90), hs_max=hs.max(), tm_mean=tm.mean(),
                dir_from=dir_from, dir_prop=(dir_from + 180) % 360,
                Pl_net=p_l.mean(),                                # >0 net drive toward NNW, <0 toward SSE
                pct_onshore=100 * (shoreward > 0).mean(), n_hours=len(df))


rows = []
for (glat, glon), df in wave_raw.items():
    for season in SEASONS:
        sub = df[df["time"].dt.month.isin(months_in(season))]
        rows.append(dict(cell=f"{glat:.2f}N,{glon:.2f}E", grid_lat=glat, grid_lon=glon, season=season, **wave_stats(sub)))
WAVE_STATS = pd.DataFrame(rows)
WAVE_STATS.to_csv(os.path.join(OUT_DIR, "era5_wave_stats.csv"), index=False)
display(WAVE_STATS.round(3))

# all cells pooled: monthly climatology and annual series
pooled = pd.concat(wave_raw.values(), ignore_index=True)
monthly = pooled.groupby(pooled["time"].dt.month)["hs"].mean()
annual = pooled.groupby(pooled["time"].dt.year)["hs"].mean()
monthly.rename_axis("month").reset_index().to_csv(os.path.join(OUT_DIR, "hs_monthly.csv"), index=False)
annual.rename_axis("year").reset_index().to_csv(os.path.join(OUT_DIR, "hs_annual.csv"), index=False)

HS_TREND = trend_stats(annual.index, annual.values)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in HS_TREND.items() if k != "ci95_per_decade"},
      "CI95/decade:", [round(c, 3) for c in HS_TREND["ci95_per_decade"]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.bar(monthly.index, monthly.values, color="steelblue")
ax1.set_xticks(range(1, 13)); ax1.set_xlabel("Month"); ax1.set_ylabel("Mean Hs (m)")
ax1.set_title("Monthly mean significant wave height")
ax2.plot(annual.index, annual.values, "o-", color="steelblue", label="Annual mean Hs")
ax2.plot(annual.index, HS_TREND["intercept"] + HS_TREND["slope_per_decade"] / 10 * annual.index, "k--",
         label=f"Sen slope {HS_TREND['slope_per_decade']:+.3f} m/decade (p = {HS_TREND['p']:.3f})")
ax2.set_xlabel("Year"); ax2.set_ylabel("Mean Hs (m)"); ax2.set_title("Annual mean Hs, all ERA5 cells pooled")
ax2.grid(alpha=0.3); ax2.legend()
save_fig(fig, "Fig_waves_climatology_trend.png")

In [ ]:
# Wave maps: ERA5 cells coloured by mean Hs, arrows = energy-weighted propagation direction
WAVE_EXTENT = (90.1, 92.6, 19.6, 22.0)
cmap = plt.get_cmap("YlGnBu")
norm = Normalize(WAVE_STATS["hs_mean"].min(), WAVE_STATS["hs_mean"].max())   # shared across seasons

fig, axes = map_axes(len(SEASONS), WAVE_EXTENT, (15, 6.5))
for i, (ax, season) in enumerate(zip(axes, SEASONS)):
    for _, r in WAVE_STATS[WAVE_STATS["season"] == season].iterrows():
        ax.add_patch(Rectangle((r.grid_lon - 0.25, r.grid_lat - 0.25), 0.5, 0.5, facecolor=cmap(norm(r.hs_mean)),
                               edgecolor="gray", alpha=0.9, zorder=1, **TR))
        ax.text(r.grid_lon, r.grid_lat - 0.12, f"{r.hs_mean:.2f} m", ha="center", va="center", fontsize=8,
                zorder=4, **TR)
        e, n = unit(r.dir_prop)                                  # FROM + 180 = direction of travel
        ax.quiver(r.grid_lon, r.grid_lat + 0.07, e, n, color="black", pivot="mid", scale=8,
                  scale_units="width", width=0.006, zorder=4, **TR)
    add_zones(ax, legend=(i == 0))
    ax.set_title(season, fontsize=11)
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=list(axes), orientation="horizontal",
             fraction=0.05, pad=0.10, label="Mean significant wave height, Hs (m)")
fig.suptitle("ERA5 mean significant wave height and wave direction, 1993–2023", fontsize=13)
save_fig(fig, "Fig_waves_seasonal.png")

In [ ]:
# Hotspot linkage (exploratory): forcing at the nearest HYCOM / ERA5 cell to each zone's offshore point
def nearest(df, lat, lon):
    d = haversine_km(lat, lon, df["lat"].values, df["lon"].values)
    i = int(np.argmin(d))
    return df.iloc[i], float(d[i])


rows = []
for name, z in ZONES.items():
    row = dict(zone=name, label=z["label"], kind=z["kind"], epr=z["epr"])
    for season in SEASONS:
        sfx = SHORT[season]
        h, h_km = nearest(HYCOM[season].dropna(subset=["u", "v", "sst"]), z["off_lat"], z["off_lon"])
        wv = WAVE_STATS[WAVE_STATS["season"] == season].rename(columns={"grid_lat": "lat", "grid_lon": "lon"})
        w, w_km = nearest(wv, z["off_lat"], z["off_lon"])
        row.update({f"cur_speed_{sfx}": h["speed"], f"cur_along_{sfx}": h["along"], f"sst_{sfx}": h["sst"],
                    f"hs_mean_{sfx}": w["hs_mean"], f"hs_p90_{sfx}": w["hs_p90"], f"Pl_net_{sfx}": w["Pl_net"],
                    f"wave_dir_from_{sfx}": w["dir_from"], f"era5_cell_{sfx}": w["cell"],
                    f"hycom_dist_km_{sfx}": h_km, f"era5_dist_km_{sfx}": w_km})
    rows.append(row)
HOTSPOT = pd.DataFrame(rows)
HOTSPOT.to_csv(os.path.join(OUT_DIR, "hotspot_forcing.csv"), index=False)
key_cols = ["label", "kind", "epr", "cur_along_sw", "cur_along_ne", "sst_ann", "hs_mean_sw", "hs_mean_ann",
            "Pl_net_sw", "Pl_net_ann", "era5_cell_ann", "hycom_dist_km_ann", "era5_dist_km_ann"]
display(HOTSPOT[key_cols].round(3))

# Spearman correlation of EPR with each variable (only if >= 4 distinct values among the 6 zones).
# With n = 6 this is purely exploratory; p-values are weak evidence.
corr_rows = []
for sfx in SHORT.values():
    for var in ["cur_speed", "cur_along", "sst", "hs_mean", "hs_p90", "Pl_net"]:
        col = f"{var}_{sfx}"
        n_distinct = HOTSPOT[col].round(6).nunique()
        if n_distinct >= 4:
            rho, p = stats.spearmanr(HOTSPOT["epr"], HOTSPOT[col])
            corr_rows.append(dict(variable=col, n_distinct=n_distinct, rho=rho, p=p, note="tested"))
        else:
            corr_rows.append(dict(variable=col, n_distinct=n_distinct, rho=np.nan, p=np.nan,
                                  note="not tested – zones share grid cells"))
CORR = pd.DataFrame(corr_rows)
CORR.to_csv(os.path.join(OUT_DIR, "hotspot_correlations.csv"), index=False)
display(CORR.round(3))

# figure: EPR, SW-monsoon alongshore current, annual longshore wave-energy-flux index
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panels = [("epr", "EPR 1992–2023 (m/yr)", "Shoreline change rate"),
          ("cur_along_sw", "Alongshore current (m s⁻¹)", "SW monsoon alongshore current\n(+ toward NNW, − toward SSE)"),
          ("Pl_net_ann", "Pl_net (arbitrary units)", "Annual longshore wave-energy-flux index\n(+ toward NNW, − toward SSE)")]
for ax, (col, ylabel, title) in zip(axes, panels):
    colours = ["red" if k == "Erosion" else "green" for k in HOTSPOT["kind"]]
    ax.bar(HOTSPOT["label"], HOTSPOT[col], color=colours, edgecolor="black")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=10); ax.grid(axis="y", alpha=0.3)
fig.suptitle("Shoreline change and offshore forcing by hotspot zone (red = erosion, green = accretion)")
save_fig(fig, "Fig_hotspot_forcing.png")

In [ ]:
# Cyclones within 250 km of the coast (IBTrACS, North Indian basin) -- a failure here never stops the notebook
CYCLONES = None
if RUN_CYCLONES:
    try:
        base = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
                "{v}/access/csv/ibtracs.NI.list.{v}.csv")
        cols = ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "WMO_WIND", "USA_WIND"]
        for version in ("v04r01", "v04r00"):
            try:
                tracks = pd.read_csv(base.format(v=version), skiprows=[1], low_memory=False, usecols=cols)
                print("IBTrACS", version, "loaded")
                break
            except Exception as err:
                print("IBTrACS", version, "failed:", str(err)[:80])
        for c in ["SEASON", "LAT", "LON", "WMO_WIND", "USA_WIND"]:
            tracks[c] = pd.to_numeric(tracks[c], errors="coerce")
        tracks = tracks[tracks["SEASON"].between(1992, 2023)].copy()
        tracks["dist_km"] = haversine_km(21.10, 92.15, tracks["LAT"], tracks["LON"])
        near = tracks[tracks["dist_km"] <= 250]
        CYCLONES = (near.groupby("SID").agg(name=("NAME", "first"), season=("SEASON", "first"),
                                            min_dist_km=("dist_km", "min"), max_wmo_wind_kt=("WMO_WIND", "max"),
                                            max_usa_wind_kt=("USA_WIND", "max"))
                    .reset_index(drop=True).sort_values("season"))
        CYCLONES.to_csv(os.path.join(OUT_DIR, "cyclones_250km.csv"), index=False)
        display(CYCLONES.round(1))

        # storm counts per 5-year period
        start = (CYCLONES["season"] - 1992) // 5 * 5 + 1992
        counts = start.value_counts().reindex(range(1992, 2024, 5), fill_value=0).sort_index()
        labels = [f"{s}–{min(s + 4, 2023)}" for s in counts.index]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.bar(labels, counts.values, color="slategray", edgecolor="black")
        ax.set_xlabel("Period"); ax.set_ylabel("Number of storms")
        ax.set_title("Cyclones passing within 250 km of Cox's Bazar–Teknaf coast")
        ax.grid(axis="y", alpha=0.3)
        save_fig(fig, "Fig_cyclones.png")
    except Exception as err:
        CYCLONES = None
        print("Cyclone step skipped:", err)

In [ ]:
# Summary and download
def circ_mean_deg(angles):
    """Mean of directions in degrees (vector mean)."""
    a = np.radians(np.asarray(angles, float))
    return float(np.degrees(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)


summary = {"period": f"{PERIOD_START} to {PERIOD_END}", "hycom": {}, "era5": {}}
for season in SEASONS:
    h = HYCOM[season]
    summary["hycom"][season] = dict(mean_speed=h["speed"].mean(), max_speed=h["speed"].max(),
                                    mean_alongshore=h["along"].mean(), mean_sst=h["sst"].mean())
    w = WAVE_STATS[WAVE_STATS["season"] == season]
    summary["era5"][season] = dict(hs_mean=w["hs_mean"].mean(), hs_p90=w["hs_p90"].mean(), tm_mean=w["tm_mean"].mean(),
                                   dir_from_mean=circ_mean_deg(w["dir_from"]), Pl_net_mean=w["Pl_net"].mean())
summary["SST_TREND"] = SST_TREND
summary["HS_TREND"] = HS_TREND
hot_cols = ["zone", "kind", "epr", "cur_speed_sw", "cur_along_sw", "cur_along_ne", "sst_ann", "hs_mean_sw", "hs_mean_ann",
            "hs_p90_ann", "Pl_net_sw", "Pl_net_ne", "Pl_net_ann"]
summary["hotspots"] = HOTSPOT[hot_cols].round(3).to_dict(orient="records")
summary["correlations"] = CORR.round(3).to_dict(orient="records")
if CYCLONES is not None:
    summary["cyclones"] = CYCLONES.round(1).to_dict(orient="records")

with open(os.path.join(OUT_DIR, "SUMMARY.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("==== SUMMARY (send to Claude) ====")
print(json.dumps(summary, indent=2, default=str))

# zip the outputs (the large raw ERA5 download cache is left out to keep the zip small)
staging = "/content/_zip_staging"
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(OUT_DIR, staging, ignore=shutil.ignore_patterns("era5_waves"))
shutil.make_archive("/content/ocean_outputs", "zip", staging)
shutil.rmtree(staging, ignore_errors=True)
try:
    from google.colab import files
    files.download("/content/ocean_outputs.zip")
except Exception as err:
    print("Automatic download failed:", err)
    print("The zip is at /content/ocean_outputs.zip (Files panel on the left).")